__Delete this cell later__:

---

idea for structuring the text in notebook: We could do something like this:
 - Start by saying "we have done x, y & Z know ...", "based on the load data we can know do x..." and etc..
 - Do the next step/task.
 - Finish each thing with *Answer:* "We found that ..."

---

Things that needs check/changes:
- Idk if this requirements.txt is correct, if it is I thing its a good idea, this could also be done in a gitworkflow file, but maybe too much for now. 
- Check if source(s) and quotes are correct
- In the cleaning data section, idk about this price outlier removal - I just said everything under 500 remove. 
---

## Imports need for notebook 

In [1678]:
# Uncomment line 2 and run this cell to install the required packages for the project.
# pip install -r requirements.txt

In [1679]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore")

In [1680]:
# Only uncomment line 2 and run this cell if the imports have changed
# pip freeze > requirements.txt

## 1. Bussiness case / Problem statement

Across the world, cars are one of the most essential mode of transportaion. In som regions more then others, which leads to a high volume of vehicle transations. For example, according to S&P Global: <br><br>
&emsp;&emsp;*"[...] 2025 US auto sales in April expected to reach 1.49 million units [...]"*<br>
&emsp;&emsp;source: [automotive-insights](https://www.spglobal.com/automotive-insights/en/blogs/2025/02/us-auto-sales-2025) (last visited: 29-04-2025). 
<br>

Where there is a high number of cars being bought and sold, they will eventually end up at used car dealerships. According to [ibisworld](https://www.ibisworld.com/united-states/number-of-businesses/used-car-dealers/1004/) (last visited: 29-04-2025): There where <br>

&emsp;&emsp;*"[...] 130,152 Used Car Dealers in the US businesses as of 2023, an decrease of -0.6% from 2022."*
<br>

Based on this we can wonder how can these used car dealerships price their cars?
What factors inpacts the pricing of a used car? And can machine learning be used to give a accurate price prediction?

## 2. Data selection and preparation:

This section will include:
- Loading the data
- Cleaning the data

This exam project is based on the dataset(s): [Car Prices Dataset](https://www.kaggle.com/datasets/sidharth178/car-prices-dataset?select=train.csv) from kaggle. 

This contains two datasets: test.csv and train.csv, where the test set dont contain price. 
These sets of data don't seem to have been pre cleaned. 
A thing to consider is that if the test.csv is to close to the train.csv, it might give some bias in the model and fit it to well, so we need to be careful with that.

### 2.1 Load data

In [1681]:
base_path = '../data/'

df_test = pd.read_csv(f'{base_path}test.csv', sep=',', header=0)
df = pd.read_csv(f'{base_path}train.csv', sep=',', header=0)

We will load the data using pandas read_csv function, where we have specified the separator to separate by sep = ',' and with header = 0 to instruct that the first row of the csv file is a header, wich will become the columns in the dataframe.

After the data has been loaded, we verify that the data has been loadded properly by sampling 5 ranomd rows of the data, by using the sample function: 

In [1682]:
df.sample(5)

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
8620,45420666,83107,1292,LAND ROVER,Discovery,2016,Jeep,Yes,Diesel,3.0 Turbo,83000 km,6.00,Tiptronic,4x4,04-May,Left wheel,Brown,12
12276,45647685,314,1646,LEXUS,GX 460,2015,Jeep,Yes,Petrol,4.6,102907 km,8.00,Automatic,4x4,04-May,Left wheel,Black,0
14273,45786062,28000,1017,BMW,520 d xDrive Luxury,2017,Sedan,Yes,Diesel,2,48578 km,4.00,Automatic,Front,>5,Left wheel,Sky blue,12
3600,45793749,4359,1285,CHEVROLET,Captiva,2007,Jeep,Yes,Diesel,2,52699 km,4.00,Automatic,Front,04-May,Left wheel,Silver,4
17656,45816496,21012,1017,FORD,Focus,2017,Hatchback,No,Petrol,2,75200 km,4.00,Tiptronic,Front,04-May,Left wheel,Silver,8


In [1683]:
df_test.sample(5)

,ID,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags,Price
3873,45631462,1451,BMW,X5,2011,Sedan,Yes,Petrol,4.4,139363 km,8,Automatic,4x4,04-May,Left wheel,Black,0,NaN
1906,45729837,831,SSANGYONG,Actyon,2017,Jeep,Yes,Petrol,1.6,40546 km,4,Automatic,Front,04-May,Left wheel,Black,4,NaN
2866,45423334,777,TOYOTA,Camry SPORT,2014,Sedan,Yes,Petrol,2.5,102300 km,4,Tiptronic,Front,04-May,Left wheel,Grey,12,NaN
5656,45754383,-,MERCEDES-BENZ,B 180,2007,Hatchback,No,Diesel,2,239000 km,4,Manual,Front,04-May,Left wheel,Silver,6,NaN
2567,45793649,966,CHEVROLET,Captiva,2009,Jeep,Yes,Diesel,2,42545 km,4,Automatic,Front,04-May,Left wheel,White,4,NaN


In [1684]:
df.shape

(19237, 18)

In [1685]:
pd.set_option('display.float_format', '{:.2f}'.format)
df.describe()

,ID,Price,Prod. year,Cylinders,Airbags
count,19237.00,19237.00,19237.00,19237.00,19237.00
mean,45576535.89,18555.93,2010.91,4.58,6.58
std,936591.42,190581.27,5.67,1.20,4.32
min,20746880.00,1.00,1939.00,1.00,0.00
25%,45698374.00,5331.00,2009.00,4.00,4.00
50%,45772308.00,13172.00,2012.00,4.00,6.00
75%,45802036.00,22075.00,2015.00,4.00,12.00
max,45816654.00,26307500.00,2020.00,16.00,16.00


We can see that the data has been loaded properly. Additionally this process also helps us to quickly get a view and idea of the data we are going to work with. 

We can first of all see that the `ID` column is not need beacuse the datafram has its own bulid in index. 
Secondly we can see that the `Doors` column looks a bit strange, beacause it containes 3 opstions: '04-May', '02-Mar' or '>5', which couold be a mistake in the data, so this has to be checked more in detail. 
Thirdly we can see that when `Levy` don't have a value its just a '-'. Lastly we can also see that the `Price` has an unrealistic min value of 1.00 and a bit outlier of 26307500.00 as max value, more then likely this should just be removed.

### 2.2 Cleaning- and Preprocessing Data

In [1686]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19237 entries, 0 to 19236
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID                19237 non-null  int64  
 1   Price             19237 non-null  int64  
 2   Levy              19237 non-null  object 
 3   Manufacturer      19237 non-null  object 
 4   Model             19237 non-null  object 
 5   Prod. year        19237 non-null  int64  
 6   Category          19237 non-null  object 
 7   Leather interior  19237 non-null  object 
 8   Fuel type         19237 non-null  object 
 9   Engine volume     19237 non-null  object 
 10  Mileage           19237 non-null  object 
 11  Cylinders         19237 non-null  float64
 12  Gear box type     19237 non-null  object 
 13  Drive wheels      19237 non-null  object 
 14  Doors             19237 non-null  object 
 15  Wheel             19237 non-null  object 
 16  Color             19237 non-null  object

We can see that there are no null values in the data; however we know that there are some columns like `Levy`, that have '-' instead of NaN.

In [1687]:
df_dash_count = df['Levy'].where(df['Levy'] == '-').count()
df_test_dash_count = df_test['Levy'].where(df_test['Levy'] == '-').count()

print(f"Number of '-' in Levy in train set: {df_dash_count}")
print(f"Number of '-' in Levy in test set: {df_test_dash_count}")

Number of '-' in Levy in train set: 5819
Number of '-' in Levy in test set: 2454


In [1688]:
levy_avg = np.sum(df['Levy'].where(df['Levy'] != '-').astype(float) / df.shape[0])
levy_test_avg = np.sum(df_test['Levy'].where(df_test['Levy'] != '-').astype(float) / df_test.shape[0])

print(f"Average Levy in train set: {levy_avg}")
print(f"Average Levy in test set: {levy_test_avg}")

df['Levy'] = df['Levy'].replace('-', int(levy_avg))
df_test['Levy'] = df_test['Levy'].replace('-', int(levy_test_avg))


Average Levy in train set: 632.5286687113374
Average Levy in test set: 644.672771376592


We have exchanged the '-' with the average ´Levy´ value, which is about 632 for the train set and 631 for the test set.

As we saw, when we used the descpribe function, the `Price` had an unrealistic min (1) and max (2.6 m) value, most likely outliers, even though max could be real. We will have a closer look at this.

In [1689]:
low_outliers_custom = df[df['Price'] < 500]
high_outliers_custom = df[df['Price'] > 500000]

high_outliers_custom

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
1225,45795524,627220,632,MERCEDES-BENZ,G 65 AMG 63AMG,2020,Jeep,Yes,Petrol,6.3 Turbo,0 km,8.00,Tiptronic,4x4,04-May,Left wheel,Black,12
8541,45761204,872946,2067,LAMBORGHINI,Urus,2019,Universal,Yes,Petrol,4,2531 km,8.00,Tiptronic,4x4,04-May,Left wheel,Black,0
16983,45812886,26307500,632,OPEL,Combo,1999,Goods wagon,No,Diesel,1.7,99999 km,4.00,Manual,Front,02-Mar,Left wheel,Blue,0


In [1690]:
df = df[df['Price'] > 500]
df = df.drop(index=16983)

We have come to that conclusion that the 2.6m car is more a mistake then the real price, and that cars under 500 should be removed.

In [1691]:
df['Doors'].value_counts()

Doors
04-May    16695
02-Mar      757
>5          119
Name: count, dtype: int64

In [1692]:
df['Doors'] = df['Doors'].replace({
    '04-May': '4',
    '02-Mar': '2',
    '>5': '5'
})

df_test['Doors'] = df_test['Doors'].replace({
    '04-May': '4',
    '02-Mar': '2',
    '>5': '5'
})

As we mentioned before in 2.1 load data, we observed that `Doors` had a wrong format, by mistake. We have used .value_counts() to get a full view over the different types of values present in the dataframe, so we can transform these into more usable values.

In [1693]:
df['Engine volume'].value_counts()

Engine volume
2            3706
2.5          2051
1.8          1569
1.6          1413
1.5          1203
             ... 
5.4 Turbo       1
0.3 Turbo       1
5.2             1
5.8             1
1.1 Turbo       1
Name: count, Length: 107, dtype: int64

Right know this `Engine volume` is an object in the dataframe, but could just be a float; however these values can contain a non-numeric character, beacuse som engines are turbo engines. So instead of just converting it to float, and removing the information about if a engine it turbo or not, we will first look at how many there are turbo engines to see if it is worth to keep this information.

In [1694]:
turbo_conut = df['Engine volume'].where(df['Engine volume'].str.contains("Turbo")).count()
print(f"Number of Turbo engines: {turbo_conut}, which is {(turbo_conut / df.shape[0]) * 100:.2f}% of the dataset")

Number of Turbo engines: 1896, which is 10.79% of the dataset


We can see that there are 1931 turbo engines, which is 10% ish of the cars engines, that are turbo, so we decided to keep this information, due to i maybe have inpact on the price.

In [1695]:
df['turbo'] = df['Engine volume'].apply(lambda x: 1 if 'Turbo' in x else 0)
df['Engine volume'] = df['Engine volume'].str.replace('Turbo', '').astype(float)

We have made a new column that representes if a engine is turbo or not (1 = turbo, 0 = not turbo), and we have removed 'Turbo' at the end of each occasion and lastly converted the `Engine volume` to a float, by using `.astype(float)`.

In [1696]:
df['Manufacturer'].value_counts()

Manufacturer
HYUNDAI          3629
TOYOTA           3195
MERCEDES-BENZ    1814
CHEVROLET        1006
FORD              991
                 ... 
LAMBORGHINI         1
PONTIAC             1
SATURN              1
ASTON MARTIN        1
GREATWALL           1
Name: count, Length: 64, dtype: int64

In [1697]:
df['Leather interior'].value_counts()

Leather interior
Yes    12542
No      5029
Name: count, dtype: int64

In [1698]:
df['Leather interior'] = df['Leather interior'].replace({'No': 0, 'Yes': 1})
df['Leather interior'].value_counts()

Leather interior
1    12542
0     5029
Name: count, dtype: int64

#### Transformation of data types to numeric

In [1699]:
df['Mileage'] = df['Mileage'].str.replace(' km', '')
df_test['Mileage'] = df_test['Mileage'].str.replace(' km', '')

df.rename(columns={'Mileage': 'Mileage_km'}, inplace=True)
df_test.rename(columns={'Mileage': 'Mileage_km'}, inplace=True)


to_numeric = ['Prod. year', 'Mileage_km', 'Levy', 'Doors', 'Leather interior']

df[to_numeric] = df[to_numeric].apply(pd.to_numeric, errors='coerce')
df_test[to_numeric] = df_test[to_numeric].apply(pd.to_numeric, errors='coerce')

Every value in the `Mileage` column ends with 'km', we would rather just have the numric value, so it can be easier to work with, instead of a string. Therfore have we first of all removed the 'km' for each value and changed the name of the column to `Mileage_km`.

The columns: `Prod. year`, `Mileage_km`, `Levy`, `Doors` have been changed to numeric values, by applying the function pd.to_numeric() to each column, so that these columns can be used in training the models and making predictions.

In [1700]:
df.drop(columns=['ID'], inplace=True)
df_test.drop(columns=['ID', 'Price'], inplace=True)

As stated previously, we remove `ID` because we don't need two indexes, due to dataframe has an build in index and an additional is redundant.

#### 2.2.1 Verifying data after cleaning

Just in case, we check if the values have been drop, have missing values and that we have the right data types, so we don't end with unexpected values in the data. We do this as we did after loading the data, by randomly sampling 5 rows from the datasets and using the info function to see null values and data types.

In [1701]:
df.sample(5)

,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage_km,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags,turbo
16145,10036,475,CHEVROLET,Cruze,2012,Sedan,0,Petrol,1.40,148000,4.00,Tiptronic,Front,4,Left wheel,Grey,10,0
7086,19130,738,CHEVROLET,Cruze LT,2017,Sedan,0,Petrol,1.40,104000,6.00,Automatic,Front,4,Left wheel,Silver,8,1
5137,39201,1091,HYUNDAI,H1,2016,Universal,1,Diesel,2.50,182821,4.00,Automatic,Front,4,Left wheel,Silver,4,0
11398,31047,1077,HYUNDAI,Veloster,2019,Hatchback,0,Petrol,2.00,12000,4.00,Tiptronic,Front,4,Left wheel,White,12,0
17128,19121,531,HYUNDAI,Elantra,2012,Sedan,1,Petrol,1.60,112243,4.00,Automatic,Front,4,Left wheel,White,4,0


In [1702]:
df_test.sample(5)

,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage_km,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
7684,642,SSANGYONG,Korando,2012,Jeep,NaN,Diesel,2,186000,4,Automatic,Front,4,Left wheel,White,4
2568,528,HYUNDAI,Elantra,2014,Sedan,NaN,Petrol,1.6,60951,4,Automatic,Front,4,Left wheel,Silver,4
7436,644,MERCEDES-BENZ,C 200 KOMPRESSOR,2001,Coupe,NaN,LPG,2,173000,4,Tiptronic,Rear,2,Right-hand drive,Grey,6
4418,753,KIA,Sportage,2012,Jeep,NaN,Petrol,2.4,229848,4,Automatic,4x4,4,Left wheel,Grey,12
421,644,FIAT,500 Lounge,2012,Coupe,NaN,Petrol,1.4,121000,4,Manual,Front,2,Left wheel,Black,4


In [1703]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8245 entries, 0 to 8244
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Levy              8245 non-null   int64  
 1   Manufacturer      8245 non-null   object 
 2   Model             8245 non-null   object 
 3   Prod. year        8245 non-null   int64  
 4   Category          8245 non-null   object 
 5   Leather interior  0 non-null      float64
 6   Fuel type         8245 non-null   object 
 7   Engine volume     8245 non-null   object 
 8   Mileage_km        8245 non-null   int64  
 9   Cylinders         8245 non-null   int64  
 10  Gear box type     8245 non-null   object 
 11  Drive wheels      8245 non-null   object 
 12  Doors             8245 non-null   int64  
 13  Wheel             8245 non-null   object 
 14  Color             8245 non-null   object 
 15  Airbags           8245 non-null   int64  
dtypes: float64(1), int64(6), object(9)
memory 

We can know comfirm that the values have infact been droped, have the right data types and that we have no missing values.

Lastly we save the cleaned data to a new csv file, so we can use it for the other part of the this project.

In [ ]:
df.to_csv(f'{base_path}dataset_cleaned.csv', sep=',', index=False)

(17571, 18)

## 3 Data exploration and visualization:

This section will include:
- x
- y



--- 
Look at the data and try to understand it. What are the most important features?

Make some visualizations to understand the data better, by showing the distribution of the features and the target variable.

Try to find correlations between the features and the target variable, by making scatter plots and correlation matrices.

etc ...

## 4. Model Selection and training:

This section will include:
- x
- y

### 4.1 The selected model(s)

We have selected the following models for this project:
- Multiple Linear Regression
- Model 2

The reason for the selected models is as follows:
- Multiple Linear Regression: ..
- Model 2: This model is selected because ...

We are going to be using the following error metrics to evalaute the models:
- x
- y
- z

Additionaly we are going to be useing the grid search approach for ...

### 4.2 Model training and validation

Based on the selected models we have trained the models using the training data. The training process is as follows:

"describe the training process here, that we are going to use and why that makes sence"

### 4.3 Model evaluation